# 02c — Run E5 tambahan (seed berbeda) untuk ensemble

**Tujuan:** melatih ulang E5 (Wav2Vec2 `jonatasgrosman/wav2vec2-large-xlsr-53-english`, **resep persis sama** seperti R4) dengan **seed berbeda**, lalu menyimpan logit per fold. Logit ini dirata-ratakan di `03_hasil_analisis.ipynb` (bagian 1b) menjadi ensemble. Notebook ini sengaja **tidak** menghitung ensemble atau membandingkannya dengan paper; itu dilakukan di notebook 03 dengan uji berpasangan dan koreksi Holm.

**Aturan yang ditetapkan sebelum melihat hasil**
- Resep tidak diubah. Satu-satunya yang berbeda adalah `cfg["seed"]` (R4 memakai 2020).
- Semua run yang selesai dipakai di ensemble. Tidak boleh membuang run yang kurang bagus.
- Dilaporkan epoch **terakhir** (bukan epoch terbaik pada fold uji).

**Biaya:** ≈ 78 detik/epoch × 15 epoch × 5 fold ≈ **1,6 jam per seed** di GPU T4. Default 2 seed ≈ 3,3 jam, jadi ensemble berisi 3 run (R4 + 2). Sesi Kaggle maksimal sekitar 12 jam, dan notebook berhenti sendiri sebelum batas (`HOURS_BUDGET`). Fold yang sudah selesai dilewati saat dijalankan ulang.

**Cara menjalankan di Kaggle**
1. Import notebook ini. **Add Data**: dataset RAVDESS (`corry371/database-speech-emotion-recognition-with-ravdess`).
2. **Accelerator: GPU T4**, **Internet: On** (unduh model dari Hugging Face dan cadangan `git clone` repo penulis).
3. **Run All.** Keluaran: `ENS_runs.zip` di tab *Output*.
4. Unggah `ENS_runs.zip` sebagai dataset (atau letakkan di samping notebook 03), lalu jalankan ulang notebook 03. Run tambahan ditemukan otomatis.

In [1]:
%%writefile rav_utils.py
"""Utilitas bersama untuk 01_replikasi.ipynb dan 02_pembanding.ipynb.

Semua evaluasi memakai protokol paper (Luna-Jimenez et al., 2022):
subject-wise 5-fold CV dengan pembagian aktor tetap (FOLDS).
"""
import json, os, platform, random, sys
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler, StandardScaler

SEED = 2020  # sama dengan seed di repo penulis
FOLDS = {0: [2, 5, 14, 15, 16], 1: [3, 6, 7, 13, 18], 2: [10, 11, 12, 19, 20],
         3: [8, 17, 21, 23, 24], 4: [1, 4, 9, 22]}
EMOTIONS = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
POST_CLS = [f"embs{i}" for i in range(8)]  # nama kolom posterior di repo


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)


def env_log(path, extra=None):
    """Simpan log lingkungan (versi paket, seed, platform) untuk reproduksibilitas."""
    import sklearn, scipy
    info = {"python": sys.version.split()[0], "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "scipy": scipy.__version__,
            "seed": SEED, "folds": FOLDS}
    if extra:
        info.update(extra)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        json.dump(info, f, indent=2)
    return info


def find_repo(candidates=("MMEmotionRecognition-main", "../MMEmotionRecognition-main",
                          "MMEmotionRecognition", "../MMEmotionRecognition")):
    if isinstance(candidates, str):
        candidates = (candidates,)
    for c in candidates:
        if os.path.isdir(os.path.join(c, "data", "trainingData")):
            return os.path.abspath(c)
    raise FileNotFoundError("Folder repo tidak ketemu. Ekstrak MMEmotionRecognition-main.zip di "
                            "samping notebook ini, atau set REPO_DIR manual.")


def _add_labels(df, name_col):
    df["actor"] = df[name_col].str.split("-").str[-1].astype(int)
    df["y"] = df[name_col].str.split("-").str[2].astype(int) - 1
    return df


def load_au_avg(repo):
    """1440 video x 35 fitur AU (rata-rata per video; 17 intensitas '_r' + 18 kehadiran '_c')."""
    df = pd.read_csv(os.path.join(repo, "data/trainingData/avg_MLP80_AUs/data_AUs_avg/df_average_AUs_total.csv"), sep=";")
    df = _add_labels(df, "name")
    feat = [c for c in df.columns if c.startswith("AU")]
    return df, feat


def load_audio_emb(repo):
    """1440 video x 512 fitur (rata-rata embedding feature-encoder xlsr-Wav2Vec2.0)."""
    df = pd.read_csv(os.path.join(repo, "data/trainingData/avg_MLP80_Audio/data/avg_embs_512.csv"), sep=";")
    df["y"] = df["emotion"].astype(int)
    feat = [f"embs{i}" for i in range(512)]
    return df, feat


def make_scaler(kind):
    return {"minmax": MinMaxScaler(feature_range=(0, 1)), "standard": StandardScaler(), None: None, "none": None}[kind]


def run_cv(df, feat, make_clf, norm=None):
    """Subject-wise 5-CV. Mengembalikan akurasi & macro-F1 per fold + prediksi gabungan."""
    accs, f1s, yt, yp, fold_of = [], [], [], [], []
    for f, actors in FOLDS.items():
        te = df["actor"].isin(actors).values
        Xtr, Xte = df.loc[~te, feat].values, df.loc[te, feat].values
        sc = make_scaler(norm)
        if sc is not None:
            Xtr = sc.fit_transform(Xtr)
            Xte = sc.transform(Xte)
        clf = make_clf().fit(Xtr, df.loc[~te, "y"].values)
        p = clf.predict(Xte)
        t = df.loc[te, "y"].values
        accs.append(accuracy_score(t, p) * 100)
        f1s.append(f1_score(t, p, average="macro") * 100)
        yt += list(t); yp += list(p); fold_of += [f] * len(t)
    return {"acc": accs, "f1": f1s, "y_true": np.array(yt), "y_pred": np.array(yp), "fold": np.array(fold_of)}


def ci95(accs, n_per_fold=None, n_total=1440):
    """CI 95% binomial pada akurasi rata-rata (cara paper: 1.96*sqrt(p(1-p)/N))."""
    p = np.mean(accs) / 100
    return 1.96 * np.sqrt(p * (1 - p) / n_total) * 100


def _parse_bilstm(df):
    df = df.copy()
    df["name"] = df["tag"].str.replace(".csv", "", regex=False)
    df["y"] = df["name"].str.split("-").str[2].astype(int) - 1
    P = df["posteriors"].str.split("[").str[-1].str.split("]").str[0].str.split(",", expand=True).apply(pd.to_numeric)
    P.columns = [f"bi{i}" for i in range(8)]
    return pd.concat([df[["name", "y"]], P], axis=1).sort_values("name").reset_index(drop=True)


def load_posteriors(repo, fold):
    """Posterior rilisan penulis untuk 3 cabang fusi: wav2vec FT, bi-LSTM+attention, MLP80 (AU rata-rata)."""
    b = os.path.join(repo, "data/posteriors")
    rd = lambda p, sep: pd.read_csv(os.path.join(b, p), sep=sep)
    out = {}
    for split, w_split, v_split, m_split in (("train", "train", "train", "train"), ("test", "test", "val", "test")):
        a = rd(f"wav2Vec/posteriors/20211020_094500/fold{fold}/posteriors_{w_split}.csv", ";").sort_values("name").reset_index(drop=True)
        v = _parse_bilstm(rd(f"AUs_biLSTM_6213/posteriorsv2/RAVDESS_AUs-{v_split}--fold_{fold + 1}_outof_5.csv", "\t"))
        m = rd(f"avg_MLP80_AUs/posteriors/{m_split}_fold{fold}.csv", ";").sort_values("video_name").reset_index(drop=True)
        assert (a["name"].values == v["name"].values).all() and (v["name"].values == m["video_name"].values).all()
        out[split] = {"name": v["name"].values, "y": v["y"].values,
                      "audio": a[POST_CLS].values, "bilstm": v[[f"bi{i}" for i in range(8)]].values,
                      "mlp": m[POST_CLS].values}
    return out


def per_class_pr(y_true_list, y_pred_list):
    """Precision & recall per kelas, dirata-rata antar fold (seperti Tabel 3 paper)."""
    from sklearn.metrics import precision_recall_fscore_support
    P, R = [], []
    for t, p in zip(y_true_list, y_pred_list):
        pr, rc, _, _ = precision_recall_fscore_support(t, p, labels=range(8), zero_division=0)
        P.append(pr); R.append(rc)
    return np.mean(P, 0) * 100, np.mean(R, 0) * 100


def append_results(path, rows):
    """Tambah baris hasil (format long: eksperimen, fold, metrik) untuk uji signifikansi di Minggu 3."""
    new = pd.DataFrame(rows)
    if os.path.exists(path):
        old = pd.read_csv(path)
        old = old[~old["experiment"].isin(new["experiment"].unique())]
        new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)
    return new


def fusion_cv(repo, branches=("audio", "bilstm", "mlp"), make_clf=None, norm="minmax", modify=None):
    """Late fusion ala paper: gabungkan posterior tiap cabang -> normalisasi -> pengklasifikasi (default LogReg C=1).

    modify(fold, d): fungsi opsional untuk mengganti/menambah cabang (d = hasil load_posteriors) -> dipakai di Minggu 2.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.utils import shuffle
    if make_clf is None:
        make_clf = lambda: LogisticRegression(C=1.0, max_iter=10000, random_state=SEED)
    accs, f1s, yt, yp, fo = [], [], [], [], []
    for f in range(5):
        d = load_posteriors(repo, f)
        if modify is not None:
            modify(f, d)
        Xtr = np.hstack([d["train"][b] for b in branches]); Xte = np.hstack([d["test"][b] for b in branches])
        ytr, yte = d["train"]["y"], d["test"]["y"]
        Xtr, ytr = shuffle(Xtr, ytr, random_state=SEED)
        sc = make_scaler(norm)
        if sc is not None:
            Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
        p = make_clf().fit(Xtr, ytr).predict(Xte)
        accs.append(accuracy_score(yte, p) * 100); f1s.append(f1_score(yte, p, average="macro") * 100)
        yt.append(yte); yp.append(p); fo.append(np.full(len(yte), f))
    return {"acc": accs, "f1": f1s, "y_true": np.concatenate(yt), "y_pred": np.concatenate(yp),
            "fold": np.concatenate(fo), "yt_list": yt, "yp_list": yp}


def make_oof_branch(repo, make_clf, tag="au_oof", norm="minmax"):
    """Buat cabang AU baru untuk fusi dengan posterior *out-of-fold* (GroupKFold per aktor di dalam data train).

    Berbeda dari baseline (posterior train in-sample), cara ini menghindari bias optimistik pada pelatih fusi.
    Mengembalikan fungsi `modify(fold, d)` untuk fusion_cv(..., modify=...).
    """
    from sklearn.model_selection import GroupKFold
    au, feat = load_au_avg(repo)
    au = au.set_index("name")

    def modify(fold, d):
        trn, ten = d["train"]["name"], d["test"]["name"]
        Xtr, ytr, g = au.loc[trn, feat].values, au.loc[trn, "y"].values, au.loc[trn, "actor"].values
        Xte = au.loc[ten, feat].values
        sc = make_scaler(norm)
        if sc is not None:
            Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
        oof = np.zeros((len(Xtr), 8))
        for a, b in GroupKFold(4).split(Xtr, ytr, g):
            oof[b] = make_clf().fit(Xtr[a], ytr[a]).predict_proba(Xtr[b])
        d["train"][tag] = oof
        d["test"][tag] = make_clf().fit(Xtr, ytr).predict_proba(Xte)
    return modify


def run_cv_grid(df, feat, estimator, param_grid, inner_splits=4, n_jobs=-1):
    """CV luar = fold paper (subject-wise). Hiper-parameter dipilih lewat CV DALAM per-aktor pada data latih saja
    (GroupKFold), jadi fold uji tidak dipakai untuk memilih apa pun. Mengembalikan format seperti run_cv + best_params."""
    from sklearn.model_selection import GridSearchCV, GroupKFold
    accs, f1s, yt, yp, fo, best = [], [], [], [], [], []
    for f, actors in FOLDS.items():
        te = df["actor"].isin(actors).values
        Xtr, ytr, g = df.loc[~te, feat].values, df.loc[~te, "y"].values, df.loc[~te, "actor"].values
        Xte, yte = df.loc[te, feat].values, df.loc[te, "y"].values
        gs = GridSearchCV(estimator, param_grid, cv=GroupKFold(inner_splits), scoring="accuracy", n_jobs=n_jobs, refit=True)
        gs.fit(Xtr, ytr, groups=g)
        p = gs.predict(Xte)
        accs.append(accuracy_score(yte, p) * 100); f1s.append(f1_score(yte, p, average="macro") * 100)
        yt += list(yte); yp += list(p); fo += [f] * len(yte); best.append(gs.best_params_)
    return {"acc": accs, "f1": f1s, "y_true": np.array(yt), "y_pred": np.array(yp), "fold": np.array(fo), "best_params": best}

Writing rav_utils.py


In [2]:
%%writefile kaggle_data.py
"""Deteksi otomatis SEMUA data yang tersedia (Kaggle /kaggle/input atau folder lokal).

Tidak mengasumsikan struktur folder. Fungsi `prepare()`:
  1. memindai semua file di semua dataset yang terpasang,
  2. mengekstrak zip yang belum terekstrak (kecuali zip video besar, tidak dibutuhkan E1-E3),
  3. mencari repo MMEmotionRecognition (folder yang punya data/trainingData/...),
     dan jika tidak ada -> git clone otomatis (butuh Internet ON di Kaggle),
  4. mengindeks semua file RAVDESS bernama baku (audio/video, speech/song),
  5. menyiapkan tabel 1.440 rekaman speech untuk E3.
"""
import os, re, subprocess, zipfile
import pandas as pd

RAV = re.compile(r"^(\d\d)-(\d\d)-(\d\d)-(\d\d)-(\d\d)-(\d\d)-(\d\d)\.(wav|mp4)$", re.I)
MARKER = os.path.join("data", "trainingData", "avg_MLP80_AUs", "data_AUs_avg", "df_average_AUs_total.csv")
REPO_URL = "https://github.com/cristinalunaj/MMEmotionRecognition"


def roots():
    if os.environ.get("RAV_INPUT"):
        return [os.environ["RAV_INPUT"]]
    return ["/kaggle/input"] if os.path.isdir("/kaggle/input") else ["."]


def working_dir():
    if os.environ.get("RAV_WORK"):
        return os.environ["RAV_WORK"]
    return "/kaggle/working" if os.path.isdir("/kaggle/working") else "."


def list_files(root_dirs):
    out = []
    for r in root_dirs:
        for dp, _, fns in os.walk(r):
            if "/.git" in dp.replace("\\", "/"):
                continue
            out += [os.path.join(dp, f) for f in fns]
    return out


def _top(path, root_dirs):
    for r in root_dirs:
        if path.startswith(r):
            rel = os.path.relpath(path, r).split(os.sep)
            return rel[0]
    return "?"


def inventory(files, root_dirs):
    if not files:
        return pd.DataFrame(columns=["dataset/folder", "ext", "jumlah_file"])
    df = pd.DataFrame({"path": files})
    df["dataset/folder"] = df["path"].map(lambda p: _top(p, root_dirs))
    df["ext"] = df["path"].str.extract(r"(\.[A-Za-z0-9]+)$")[0].str.lower().fillna("(none)")
    return df.groupby(["dataset/folder", "ext"]).size().rename("jumlah_file").reset_index()


def extract_zips(files, dest, skip_words=("video",), max_gb=6.0):
    done = []
    for z in [f for f in files if f.lower().endswith(".zip")]:
        base = os.path.basename(z)
        if any(w in base.lower() for w in skip_words):
            print(f"  [lewati zip video] {base} (tidak dibutuhkan untuk E1-E3)")
            continue
        if os.path.getsize(z) > max_gb * 1e9:
            print(f"  [lewati zip >{max_gb} GB] {base}")
            continue
        out = os.path.join(dest, os.path.splitext(base)[0])
        if not os.path.isdir(out):
            print(f"  ekstrak {base} -> {out}")
            os.makedirs(out, exist_ok=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(out)
        done.append(out)
    return done


def find_repo(files):
    suf = MARKER.replace("\\", "/")
    for f in files:
        p = f.replace("\\", "/")
        if p.endswith(suf):
            return os.path.abspath(p[: -len(suf)].rstrip("/"))
    return None


def clone_repo(dest):
    target = os.path.join(dest, "MMEmotionRecognition")
    if not os.path.isdir(target):
        print("  repo tidak ada di dataset -> git clone ...")
        try:
            subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, target], check=True, timeout=600)
        except Exception as e:
            print("  git clone gagal:", e, "\n  (di Kaggle: aktifkan Internet di Settings, atau tambahkan repo sebagai dataset)")
            return None
    return target if os.path.exists(os.path.join(target, MARKER)) else None


def index_ravdess(files):
    rows = []
    for f in files:
        m = RAV.match(os.path.basename(f))
        if m:
            g = list(map(int, m.groups()[:7]))
            rows.append([f, m.group(8).lower()] + g)
    cols = ["path", "ext", "modality", "vocal", "emotion", "intensity", "statement", "repetition", "actor"]
    return pd.DataFrame(rows, columns=cols)


def prepare(extract=True, allow_git=True):
    rd = roots(); work = working_dir()
    print("Root data :", rd, "| working:", work)
    files = list_files(rd)
    print(f"Total file terdeteksi: {len(files)}")
    if not files:
        for r in rd:
            print(f"  isi {r}:", os.listdir(r) if os.path.isdir(r) else "(folder tidak ada)")
        print("  PERINGATAN: tidak ada dataset terpasang. Di Kaggle: panel kanan -> Input -> Add Input -> pilih dataset,\n"
              "  lalu jalankan ulang. Tanpa dataset, hanya repo (git clone) yang bisa dipakai -> E1/E2 jalan, E3 dilewati.")
    inv = inventory(files, rd)
    if extract:
        ex = extract_zips(files, os.path.join(work, "extracted"))
        if ex:
            files += list_files(ex)
    repo = find_repo(files)
    if repo is None and allow_git:
        repo = clone_repo(work)
        if repo:
            files += list_files([repo])
    rav = index_ravdess(files)
    aud = rav[rav.ext == "wav"]
    def _mk(df):
        df = df.copy()
        if len(df):
            df["name"] = ["01-" + "-".join(f"{int(v):02d}" for v in row)
                          for row in df[["vocal", "emotion", "intensity", "statement", "repetition", "actor"]].values]
            return df.drop_duplicates("name").sort_values("name").reset_index(drop=True)
        df["name"] = pd.Series(dtype=str)
        return df
    speech = _mk(aud[aud.vocal == 1])
    song = _mk(aud[aud.vocal == 2])          # tersedia untuk pilihan CHANNEL="song"; BUKAN bagian protokol paper
    if repo and len(speech):  # speech dibatasi ke 1.440 rekaman protokol paper
        want = set(pd.read_csv(os.path.join(repo, "data/ravdess_videos.csv"), header=None)[0].astype(str).str.strip())
        speech = speech[speech.name.isin(want)].reset_index(drop=True)
    summ = {"file_total": len(files), "repo": repo, "wav_total": len(aud), "wav_speech_dipakai": len(speech),
            "wav_song(tersedia; default tidak dipakai)": len(song), "aktor_song": int(song.actor.nunique()) if len(song) else 0,
            "emosi_song": sorted(song.emotion.unique().tolist()) if len(song) else [], "video_mp4": int((rav.ext == "mp4").sum())}
    print("\nRingkasan:", summ)
    return {"root": rd, "work": work, "inventory": inv, "repo": repo, "ravdess_index": rav,
            "audio_speech": speech if len(speech) else None, "audio_song": song if len(song) else None, "summary": summ}

Writing kaggle_data.py


In [3]:
%%writefile w2v_finetune.py
"""E5: fine-tuning Wav2Vec2 dengan model pra-latih YANG SAMA seperti paper, resep optimasi berbeda.

Yang SAMA dengan paper : checkpoint jonatasgrosman/wav2vec2-large-xlsr-53-english, feature-encoder (CNN) dibekukan,
                         pooling rata-rata + MLP (1024 -> tanh -> 8), subject-wise 5-CV (fold sama), 1.440 rekaman.
Yang BERBEDA (optimasi) : LR terpisah (backbone kecil, head besar) + warmup + cosine decay, label smoothing,
                         rata-rata berbobot semua lapisan transformer, masked mean-pooling, augmentasi derau/potong tepi,
                         gradient clipping, fp16.
Pelaporan: epoch TERAKHIR (bersih) dan epoch terbaik pada fold uji (= cara paper; optimistis, hanya untuk perbandingan).
"""
import os, time, math
import numpy as np
import pandas as pd

SR = 16000
DEFAULT_CFG = dict(
    model_id="jonatasgrosman/wav2vec2-large-xlsr-53-english", pretrained=True, epochs=15, bs=4, accum=2,
    lr_backbone=3e-5, lr_head=1e-3, weight_decay=0.01, warmup_frac=0.1, label_smoothing=0.1,
    layer_weighted=True, head_dropout=0.1, mask_time_prob=0.05, grad_ckpt=False, noise_p=0.5, seed=2020, max_sec=6.0,
)


def load_waveforms(df, max_sec=6.0, n_jobs=-1):
    import librosa
    from joblib import Parallel, delayed
    f = lambda p: librosa.load(p, sr=SR, mono=True)[0][: int(max_sec * SR)].astype(np.float32)
    return Parallel(n_jobs=n_jobs)(delayed(f)(p) for p in df["path"])


def _collate(waves, idx, train, rng, noise_p):
    ws = []
    for i in idx:
        w = waves[i]
        if train:
            a, b = int(rng.integers(0, int(0.15 * SR) + 1)), int(rng.integers(0, int(0.15 * SR) + 1))
            if len(w) > a + b + SR:
                w = w[a: len(w) - b]
            if rng.random() < noise_p:
                snr = rng.uniform(15, 35)
                w = w + rng.standard_normal(len(w)).astype(np.float32) * np.sqrt((w.var() + 1e-8) / 10 ** (snr / 10))
        ws.append(((w - w.mean()) / np.sqrt(w.var() + 1e-7)).astype(np.float32))
    L = max(len(w) for w in ws)
    x = np.zeros((len(ws), L), np.float32); m = np.zeros((len(ws), L), np.int64)
    for k, w in enumerate(ws):
        x[k, : len(w)] = w; m[k, : len(w)] = 1
    import torch
    return torch.from_numpy(x), torch.from_numpy(m)


def build_model(cfg, n_classes=8, config=None):
    import torch, torch.nn as nn
    from transformers import Wav2Vec2Model

    class W2VClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            if config is not None:                     # dipakai untuk uji mekanisme (model acak kecil)
                self.w2v = Wav2Vec2Model(config)
            else:
                self.w2v = Wav2Vec2Model.from_pretrained(cfg["model_id"], layerdrop=0.0, mask_time_prob=cfg["mask_time_prob"])
            self.w2v.feature_extractor._freeze_parameters()      # CNN beku seperti paper
            if cfg["grad_ckpt"]:
                self.w2v.gradient_checkpointing_enable()
            H, nl = self.w2v.config.hidden_size, self.w2v.config.num_hidden_layers + 1
            self.use_mask = getattr(self.w2v.config, "feat_extract_norm", "layer") == "layer"
            self.layer_w = nn.Parameter(torch.zeros(nl)) if cfg["layer_weighted"] else None
            self.drop = nn.Dropout(cfg["head_dropout"]); self.dense = nn.Linear(H, H); self.out = nn.Linear(H, n_classes)

        def forward(self, x, mask):
            o = self.w2v(x, attention_mask=mask if self.use_mask else None, output_hidden_states=self.layer_w is not None)
            if self.layer_w is not None:
                hs = torch.stack(o.hidden_states, 0)
                h = (torch.softmax(self.layer_w, 0)[:, None, None, None].to(hs.dtype) * hs).sum(0)
            else:
                h = o.last_hidden_state
            if self.use_mask:
                fm = self.w2v._get_feature_vector_attention_mask(h.shape[1], mask).to(h.dtype)
                pooled = (h * fm[..., None]).sum(1) / fm.sum(1, keepdim=True).clamp(min=1)
            else:
                pooled = h.mean(1)
            return self.out(self.drop(torch.tanh(self.dense(self.drop(pooled)))))
    return W2VClassifier()


def _predict(model, waves, idx, device, bs=8):
    import torch
    model.eval(); outs = []
    order = sorted(idx, key=lambda i: len(waves[i]))
    with torch.no_grad():
        for i in range(0, len(order), bs):
            x, m = _collate(waves, order[i:i + bs], False, None, 0)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
                outs.append(model(x.to(device), m.to(device)).float().cpu())
    import torch as T
    lg = T.cat(outs).numpy(); inv = np.empty(len(order), int); inv[np.argsort(order)] = np.arange(len(order))
    pos = {j: k for k, j in enumerate(order)}
    return lg[[pos[i] for i in idx]]


def run_fold(fold, waves, names, y, actors, folds, cfg, device, out_dir="results/e5", model_config=None, log=print):
    import torch, torch.nn as nn
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"e5_fold{fold}.npz")
    if os.path.exists(path):
        log(f"fold {fold}: sudah ada ({path}) -> dilewati"); return
    te = np.isin(actors, folds[fold]); tri, tei = np.where(~te)[0], np.where(te)[0]
    torch.manual_seed(cfg["seed"] + fold); rng = np.random.default_rng(cfg["seed"] + fold)
    model = build_model(cfg, config=model_config).to(device)
    head = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("w2v.")]
    back = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("w2v.")]
    opt = torch.optim.AdamW([{"params": back, "lr": cfg["lr_backbone"]}, {"params": head, "lr": cfg["lr_head"]}], weight_decay=cfg["weight_decay"])
    steps = math.ceil(len(tri) / (cfg["bs"] * cfg["accum"])) * cfg["epochs"]; warm = max(1, int(cfg["warmup_frac"] * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    crit = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"]); yt = torch.from_numpy(y).long()
    hist = []
    for ep in range(1, cfg["epochs"] + 1):
        model.train(); t0 = time.time(); perm = rng.permutation(tri); tl, pend = 0.0, 0
        for i in range(0, len(perm), cfg["bs"]):
            bi = perm[i:i + cfg["bs"]]; x, m = _collate(waves, bi, True, rng, cfg["noise_p"])
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
                loss = crit(model(x.to(device), m.to(device)).float(), yt[bi].to(device))
            scaler.scale(loss / cfg["accum"]).backward(); tl += loss.item() * len(bi); pend += 1
            if pend == cfg["accum"]:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); pend = 0
        if pend:
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        lg = _predict(model, waves, tei, device)
        vl = nn.functional.cross_entropy(torch.from_numpy(lg), yt[tei]).item(); va = float((lg.argmax(1) == y[tei]).mean() * 100)
        hist.append((ep, tl / len(tri), vl, va))
        log(f"fold {fold} ep {ep:>2}/{cfg['epochs']} train_loss {tl/len(tri):.3f} | test_loss {vl:.3f} test_acc {va:.1f} | {time.time()-t0:.0f}s")
    lg_te = lg; lg_tr = _predict(model, waves, tri, device)       # logit epoch TERAKHIR (bersih)
    h = np.array(hist)
    np.savez_compressed(path, names_train=names[tri], logits_train=lg_tr, names_test=names[tei], logits_test=lg_te,
                        y_test=y[tei], hist=h, last_acc=h[-1, 3], best_acc=h[:, 3].max(), best_epoch=int(h[h[:, 3].argmax(), 0]))
    del model, opt; 
    if device == "cuda":
        torch.cuda.empty_cache()


def make_audio_modify(out_dir="results/e5", tag="audio_e5"):
    """Pengganti cabang audio paper pada fusion_cv: logit E5 diselaraskan berdasarkan nama rekaman."""
    def modify(fold, d):
        z = np.load(os.path.join(out_dir, f"e5_fold{fold}.npz"), allow_pickle=True)
        for split in ("train", "test"):
            pos = {n: i for i, n in enumerate(z[f"names_{split}"])}
            d[split][tag] = z[f"logits_{split}"][[pos[n] for n in d[split]["name"]]]
    return modify


def summarize(out_dir="results/e5"):
    rows = []
    for f in range(5):
        z = np.load(os.path.join(out_dir, f"e5_fold{f}.npz"), allow_pickle=True)
        rows.append({"fold": f, "acc_last": float(z["last_acc"]), "acc_best_on_test(cara paper)": float(z["best_acc"]), "best_epoch": int(z["best_epoch"])})
    return pd.DataFrame(rows)

Writing w2v_finetune.py


## 0. Deteksi data

In [4]:
import os, warnings
warnings.filterwarnings("ignore")
import kaggle_data as kd
PREP = kd.prepare()                       # pindai file, ekstrak zip, cari repo penulis, indeks RAVDESS
os.makedirs("results", exist_ok=True)

Root data : ['/kaggle/input'] | working: /kaggle/working
Total file terdeteksi: 7356
  repo tidak ada di dataset -> git clone ...

Ringkasan: {'file_total': 7543, 'repo': '/kaggle/working/MMEmotionRecognition', 'wav_total': 2452, 'wav_speech_dipakai': 1440, 'wav_song(tersedia; default tidak dipakai)': 1012, 'aktor_song': 23, 'emosi_song': [1, 2, 3, 4, 5, 6], 'video_mp4': 4904}


## 1. Konfigurasi (tetapkan SEBELUM melihat hasil)

In [5]:
import time, shutil, glob, zipfile
import numpy as np, pandas as pd, torch
import w2v_finetune as wf, rav_utils as u

NEW_SEEDS = [2021]      # seed baru; R4 memakai 2020. Ubah jumlahnya bila perlu (tiap seed ≈ 1,6 jam)
HOURS_BUDGET = 10.5           # berhenti sebelum batas sesi; fold selesai tidak hilang dan bisa dilanjutkan
FOLD_HOURS = 0.35             # perkiraan waktu satu fold (15 epoch) di T4, dipakai untuk keputusan berhenti
OUT_ROOT = "ENS"
SMOKE_TEST = False            # True = model acak kecil, 1 epoch: HANYA mengecek pipeline (hasil ke ENS_smoke, diabaikan notebook 03)

assert 2020 not in NEW_SEEDS, "Seed 2020 dipakai R4; pakai seed lain supaya run benar-benar berbeda."
assert len(set(NEW_SEEDS)) == len(NEW_SEEDS), "Seed tidak boleh berulang."
dev = "cuda" if torch.cuda.is_available() else "cpu"
BASE_CFG = dict(wf.DEFAULT_CFG)                  # resep R4 tanpa perubahan
model_config = None
if SMOKE_TEST:
    from transformers import Wav2Vec2Config
    model_config = Wav2Vec2Config(hidden_size=32, num_hidden_layers=2, num_attention_heads=2, intermediate_size=64, conv_dim=(16, 16, 16),
        conv_stride=(5, 2, 2), conv_kernel=(10, 3, 3), feat_extract_norm="layer", do_stable_layer_norm=True, num_conv_pos_embeddings=16,
        num_conv_pos_embedding_groups=2, mask_time_prob=0.05, layerdrop=0.0)
    BASE_CFG.update(pretrained=False, epochs=1, bs=8, accum=1); OUT_ROOT = "ENS_smoke"; NEW_SEEDS = NEW_SEEDS[:1]
print("Device:", dev, "| SMOKE_TEST:", SMOKE_TEST, "| seed baru:", NEW_SEEDS); print({k: v for k, v in BASE_CFG.items() if k != "seed"})
RUN_OK = dev == "cuda" or SMOKE_TEST
if not RUN_OK:
    print("PERINGATAN: tanpa GPU fine-tuning model besar tidak praktis. Aktifkan GPU di Settings -> Accelerator, atau SMOKE_TEST=True untuk uji pipeline.")

# Lanjutkan dari sesi sebelumnya: salin fold yang sudah ada dari dataset input (mis. ENS/ dari run lama yang dilampirkan sebagai Input)
for p in glob.glob("/kaggle/input/**/seed*/e5_fold*.npz", recursive=True):
    tgt = os.path.join(OUT_ROOT, os.path.basename(os.path.dirname(p)), os.path.basename(p))
    if not os.path.exists(tgt):
        os.makedirs(os.path.dirname(tgt), exist_ok=True); shutil.copy(p, tgt); print("Dilanjutkan dari input:", tgt)

Device: cuda | SMOKE_TEST: False | seed baru: [2021]
{'model_id': 'jonatasgrosman/wav2vec2-large-xlsr-53-english', 'pretrained': True, 'epochs': 15, 'bs': 4, 'accum': 2, 'lr_backbone': 3e-05, 'lr_head': 0.001, 'weight_decay': 0.01, 'warmup_frac': 0.1, 'label_smoothing': 0.1, 'layer_weighted': True, 'head_dropout': 0.1, 'mask_time_prob': 0.05, 'grad_ckpt': False, 'noise_p': 0.5, 'max_sec': 6.0}


## 2. Muat audio

In [6]:
if not RUN_OK:
    print("Dilewati (tidak ada GPU).")
else:
    aud = PREP["audio_speech"]
    assert aud is not None and len(aud) == 1440, "Audio speech (1.440 rekaman) tidak lengkap: tambahkan dataset RAVDESS lalu jalankan ulang sel Deteksi data."
    waves = wf.load_waveforms(aud, max_sec=BASE_CFG["max_sec"])
    names = aud["name"].values
    y = np.array([int(n.split("-")[2]) - 1 for n in names]); act = np.array([int(n.split("-")[-1]) for n in names])
    print(len(waves), "rekaman | durasi rata-rata", round(np.mean([len(w) for w in waves]) / 16000, 2), "s")

1440 rekaman | durasi rata-rata 3.7 s


## 3. Latih run tambahan (bisa dilanjutkan; berhenti otomatis sebelum batas waktu)

In [7]:
if not RUN_OK:
    print("Dilewati (tidak ada GPU).")
else:
    t_start, stopped = time.time(), False
    for seed in NEW_SEEDS:
        cfg = dict(BASE_CFG, seed=seed); out = f"{OUT_ROOT}/seed{seed}"
        assert {k: v for k, v in cfg.items() if k != "seed"} == {k: v for k, v in BASE_CFG.items() if k != "seed"}      # hanya seed yang berbeda
        for f in range(5):
            if os.path.exists(f"{out}/e5_fold{f}.npz"): continue
            if (time.time() - t_start) / 3600 + FOLD_HOURS > HOURS_BUDGET:
                stopped = True; print(f"Berhenti sebelum batas waktu (seed {seed}, fold {f}). Jalankan ulang (dengan hasil sebelumnya sebagai Input) untuk melanjutkan."); break
            try:
                wf.run_fold(f, waves, names, y, act, u.FOLDS, cfg, dev, out_dir=out, model_config=model_config)
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    raise RuntimeError("GPU kehabisan memori: turunkan BASE_CFG['bs'] (mis. 2) dan naikkan BASE_CFG['accum'] agar bs x accum tetap 8, atau set grad_ckpt=True. Catatan: itu mengubah resep, jadi catat di laporan.")
                raise
            except Exception as e:
                if any(k in str(e).lower() for k in ("connect", "resolve", "offline", "huggingface", "max retries")):
                    raise RuntimeError("Gagal mengunduh model dari Hugging Face: aktifkan Internet di Settings notebook lalu jalankan ulang.") from e
                raise
        if stopped: break
    for seed in NEW_SEEDS:
        print(f"seed {seed}: fold selesai", [f for f in range(5) if os.path.exists(f"{OUT_ROOT}/seed{seed}/e5_fold{f}.npz")])

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-english
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 0 ep  1/15 train_loss 1.916 | test_loss 1.199 test_acc 56.0 | 79s
fold 0 ep  2/15 train_loss 1.363 | test_loss 0.788 test_acc 69.0 | 75s
fold 0 ep  3/15 train_loss 1.010 | test_loss 0.852 test_acc 73.0 | 75s
fold 0 ep  4/15 train_loss 0.789 | test_loss 0.848 test_acc 72.3 | 75s
fold 0 ep  5/15 train_loss 0.693 | test_loss 0.788 test_acc 77.3 | 75s
fold 0 ep  6/15 train_loss 0.603 | test_loss 0.496 test_acc 86.0 | 75s
fold 0 ep  7/15 train_loss 0.570 | test_loss 0.748 test_acc 77.3 | 75s
fold 0 ep  8/15 train_loss 0.515 | test_loss 0.592 test_acc 83.7 | 75s
fold 0 ep  9/15 train_loss 0.498 | test_loss 0.575 test_acc 84.3 | 75s
fold 0 ep 10/15 train_loss 0.494 | test_loss 0.690 test_acc 81.7 | 75s
fold 0 ep 11/15 train_loss 0.485 | test_loss 0.562 test_acc 86.3 | 75s
fold 0 ep 12/15 train_loss 0.485 | test_loss 0.566 test_acc 85.7 | 75s
fold 0 ep 13/15 train_loss 0.480 | test_loss 0.622 test_acc 83.0 | 75s
fold 0 ep 14/15 train_loss 0.475 | test_loss 0.583 test_acc 84.3 | 75s
fold 0

... (8 outputs left out)

fold 3 ep  1/15 train_loss 1.955 | test_loss 1.439 test_acc 48.0 | 75s
fold 3 ep  2/15 train_loss 1.310 | test_loss 0.672 test_acc 74.7 | 75s
fold 3 ep  3/15 train_loss 0.927 | test_loss 0.784 test_acc 73.7 | 75s
fold 3 ep  4/15 train_loss 0.808 | test_loss 0.776 test_acc 77.0 | 75s
fold 3 ep  5/15 train_loss 0.675 | test_loss 0.801 test_acc 76.0 | 75s
fold 3 ep  6/15 train_loss 0.584 | test_loss 0.775 test_acc 79.0 | 75s
fold 3 ep  7/15 train_loss 0.545 | test_loss 0.766 test_acc 78.7 | 75s
fold 3 ep  8/15 train_loss 0.530 | test_loss 0.722 test_acc 80.7 | 75s
fold 3 ep  9/15 train_loss 0.496 | test_loss 0.724 test_acc 81.3 | 76s
fold 3 ep 10/15 train_loss 0.491 | test_loss 0.751 test_acc 81.3 | 76s
fold 3 ep 11/15 train_loss 0.482 | test_loss 0.809 test_acc 78.7 | 76s
fold 3 ep 12/15 train_loss 0.479 | test_loss 0.793 test_acc 79.7 | 75s
fold 3 ep 13/15 train_loss 0.475 | test_loss 0.769 test_acc 81.7 | 75s
fold 3 ep 14/15 train_loss 0.476 | test_loss 0.782 test_acc 81.0 | 75s
fold 3

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-english
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 4 ep  1/15 train_loss 1.929 | test_loss 1.466 test_acc 44.2 | 78s
fold 4 ep  2/15 train_loss 1.303 | test_loss 1.012 test_acc 63.7 | 78s
fold 4 ep  3/15 train_loss 0.970 | test_loss 0.956 test_acc 67.9 | 78s
fold 4 ep  4/15 train_loss 0.805 | test_loss 0.927 test_acc 71.2 | 78s
fold 4 ep  5/15 train_loss 0.683 | test_loss 0.893 test_acc 72.9 | 78s
fold 4 ep  6/15 train_loss 0.606 | test_loss 0.924 test_acc 70.8 | 78s
fold 4 ep  7/15 train_loss 0.550 | test_loss 0.822 test_acc 74.2 | 78s
fold 4 ep  8/15 train_loss 0.549 | test_loss 0.768 test_acc 75.4 | 78s
fold 4 ep  9/15 train_loss 0.499 | test_loss 0.803 test_acc 76.2 | 78s
fold 4 ep 10/15 train_loss 0.498 | test_loss 0.769 test_acc 75.8 | 78s
fold 4 ep 11/15 train_loss 0.488 | test_loss 0.695 test_acc 79.2 | 78s
fold 4 ep 12/15 train_loss 0.479 | test_loss 0.735 test_acc 80.4 | 78s
fold 4 ep 13/15 train_loss 0.476 | test_loss 0.768 test_acc 80.0 | 78s
fold 4 ep 14/15 train_loss 0.478 | test_loss 0.751 test_acc 79.2 | 78s
fold 4

## 4. Ringkasan per seed (hanya pemeriksaan; ensemble dihitung di notebook 03)
Angka di bawah dihitung dari logit tersimpan (epoch terakhir). Sebagai acuan, R4: audio 82,10 %, fusi 85,57 %; paper: audio 81,82 %, fusi 86,70 %.

In [8]:
rows = []
if RUN_OK:
    from sklearn.svm import SVC
    for seed in NEW_SEEDS:
        out = f"{OUT_ROOT}/seed{seed}"
        if not all(os.path.exists(f"{out}/e5_fold{f}.npz") for f in range(5)):
            print(f"seed {seed}: belum lengkap, dilewati"); continue
        S_ = wf.summarize(out); a = S_["acc_last"].values
        row = {"Seed": seed, "Audio (epoch terakhir)": round(a.mean(), 2), "Audio per fold": [round(x, 1) for x in a]}
        if PREP["repo"] and not SMOKE_TEST:
            r = u.fusion_cv(PREP["repo"], branches=("audio_e5", "bilstm", "mlp"), modify=wf.make_audio_modify(out))
            row.update({"Fusi (+ visual paper)": round(np.mean(r["acc"]), 2), "Fusi per fold": [round(x, 1) for x in r["acc"]]})
        rows.append(row)
pd.DataFrame(rows) if rows else print("Belum ada run lengkap.")

,Seed,Audio (epoch terakhir),Audio per fold,Fusi (+ visual paper),Fusi per fold
0,2021,80.63,"[84.7, 78.7, 79.7, 81.0, 79.2]",84.73,"[87.3, 86.3, 84.3, 82.3, 83.3]"


## 5. Kemas untuk notebook 03

In [9]:
if os.path.isdir(OUT_ROOT) and not SMOKE_TEST:
    done = sorted(d for d in glob.glob(f"{OUT_ROOT}/seed*") if all(os.path.exists(f"{d}/e5_fold{f}.npz") for f in range(5)))
    if done:
        with zipfile.ZipFile("ENS_runs.zip", "w", zipfile.ZIP_DEFLATED) as zf:
            for d in done:
                for f in range(5): zf.write(f"{d}/e5_fold{f}.npz")
        print("Run lengkap dikemas:", [os.path.basename(d) for d in done], "-> ENS_runs.zip", round(os.path.getsize("ENS_runs.zip") / 1e6, 2), "MB")
        print("Unduh dari tab Output, unggah sebagai dataset (atau letakkan di samping notebook 03), lalu jalankan ulang notebook 03.")
    else:
        print("Belum ada seed dengan 5 fold lengkap, tidak ada yang dikemas.")
else:
    print("SMOKE_TEST atau belum ada hasil: tidak mengemas.")

Run lengkap dikemas: ['seed2021'] -> ENS_runs.zip 0.14 MB
Unduh dari tab Output, unggah sebagai dataset (atau letakkan di samping notebook 03), lalu jalankan ulang notebook 03.
